# 1. Environment Setup & Data Access
Mounting Google Drive and initializing workspace paths.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: mount failed

# 2. Dependency Management
Installing specialized libraries for imbalanced learning, gradient boosting, and hyperparameter optimization.

In [ ]:
!pip install -q optuna lightgbm xgboost joblib

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import time
import tempfile
import tracemalloc
import joblib
import warnings
warnings.filterwarnings('ignore')

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

print("All imports successful")
print(f"XGBoost version:  {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Optuna version:   {optuna.__version__}")

# 3. Path Configuration
Defining global constants for data assets and model storage to ensure reproducibility.

In [ ]:
BASE_PATH     = "/content/drive/MyDrive/Network Intrusion Detection"
FEATURES_PATH = os.path.join(BASE_PATH, "features")
MODELS_PATH   = os.path.join(BASE_PATH, "models")
os.makedirs(MODELS_PATH, exist_ok=True)

TRAIN_PATH    = os.path.join(FEATURES_PATH, "train.csv")
TEST_PATH     = os.path.join(FEATURES_PATH, "test.csv")
FEATURES_JSON = os.path.join(FEATURES_PATH, "selected_features.json")

print("Paths configured:")
print(f"  Train:    {TRAIN_PATH}")
print(f"  Test:     {TEST_PATH}")
print(f"  Features: {FEATURES_JSON}")
print(f"  Models:   {MODELS_PATH}")

# Verify all files exist before we do anything else
!find /content/drive -name "train.csv"
for p in [TRAIN_PATH, TEST_PATH, FEATURES_JSON]:
    exists = os.path.exists(p)
    print(f"  {'✓' if exists else '✗ MISSING'} {os.path.basename(p)}")

# 4. Data Loading and Preprocessing
Loading the processed feature set and encoding labels for compatibility with ensemble models.

In [ ]:
# Load feature list
with open(FEATURES_JSON) as f:
    selected_features = json.load(f)

print(f"Selected features ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f}")

# Load train and test
print("\nLoading train.csv...")
train = pd.read_csv(TRAIN_PATH)
print(f"  Shape: {train.shape}")

print("Loading test.csv...")
test = pd.read_csv(TEST_PATH)
print(f"  Shape: {test.shape}")

# Split features and labels
X_train = train[selected_features]
y_train = train['Attack Type']
X_test  = test[selected_features]
y_test  = test['Attack Type']

# Encode labels to integers (required by XGBoost)
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f"\nClasses ({len(le.classes_)}):")
for i, cls in enumerate(le.classes_):
    train_count = (y_train == cls).sum()
    test_count  = (y_test  == cls).sum()
    print(f"  {i}: {cls:<20} train={train_count:>7,}  test={test_count:>6,}")

print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nAny NaN in train: {X_train.isnull().sum().sum()}")
print(f"Any NaN in test:  {X_test.isnull().sum().sum()}")

# 5. Benchmarking Framework
Defining a standardized evaluation suite to measure inference latency, memory footprint, and classification performance.

In [ ]:
def benchmark_model(name, model, X_test, y_test_enc, le,
                    source="our_model", n_inference_runs=1000):
    """
    Standard benchmark matching ref1/ref2 JSON schema exactly.
    y_test_enc: integer-encoded labels
    le: LabelEncoder — used to decode back to string labels
    """
    results = {"model": name, "source": source}

    # 1. Model size on disk (KB)
    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        tmp_path = f.name
    joblib.dump(model, tmp_path, compress=0)
    results["model_size_kb"] = os.path.getsize(tmp_path) / 1024
    os.unlink(tmp_path)

    # 2. Warm-up pass
    _ = model.predict(X_test[:10])

    # 3. Single-sample latency — median of n_inference_runs
    single = X_test[:1]
    latencies = []
    for _ in range(n_inference_runs):
        t0 = time.perf_counter()
        model.predict(single)
        latencies.append((time.perf_counter() - t0) * 1000)
    results["latency_single_ms"] = float(np.median(latencies))
    results["latency_p95_ms"]    = float(np.percentile(latencies, 95))

    # 4. Batch throughput on full test set
    t0 = time.perf_counter()
    y_pred_enc = model.predict(X_test)
    elapsed = time.perf_counter() - t0
    results["throughput_samples_per_sec"] = len(X_test) / elapsed
    results["batch_inference_time_s"]     = elapsed

    # 5. Peak memory during inference
    tracemalloc.start()
    _ = model.predict(X_test[:1000])
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    results["inference_peak_memory_mb"] = peak / (1024 ** 2)

    # 6. Classification metrics (weighted)
    avg = "weighted"
    results["accuracy"]    = float(accuracy_score(y_test_enc, y_pred_enc))
    results["precision"]   = float(precision_score(y_test_enc, y_pred_enc,
                                   average=avg, zero_division=0))
    results["recall"]      = float(recall_score(y_test_enc, y_pred_enc,
                                   average=avg, zero_division=0))
    results["f1_weighted"] = float(f1_score(y_test_enc, y_pred_enc,
                                   average=avg, zero_division=0))

    # 7. Per-class F1 (important for minority classes)
    per_class_f1 = f1_score(y_test_enc, y_pred_enc,
                             average=None, zero_division=0)
    results["per_class_f1"] = {
        le.classes_[i]: round(float(s), 4)
        for i, s in enumerate(per_class_f1)
    }

    # 8. Print summary
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Model size:   {results['model_size_kb']:>10.1f} KB  "
          f"({results['model_size_kb']/1024:.2f} MB)")
    print(f"  Accuracy:     {results['accuracy']:>10.4f}")
    print(f"  F1 weighted:  {results['f1_weighted']:>10.4f}")
    print(f"  Latency:      {results['latency_single_ms']:>10.3f} ms  "
          f"(p95: {results['latency_p95_ms']:.3f} ms)")
    print(f"  Throughput:   {results['throughput_samples_per_sec']:>10,.0f} samples/s")
    print(f"  Infer memory: {results['inference_peak_memory_mb']:>10.3f} MB")
    print(f"\n  Per-class F1:")
    for cls, score in results["per_class_f1"].items():
        bar = "█" * int(score * 20)
        print(f"    {cls:<20} {score:.4f}  {bar}")

    return results

print("Benchmark function ready.")

# 6. Baseline Model: Decision Tree
Performing a depth sweep to find the optimal trade-off between model complexity and performance.

In [ ]:
print("Decision Tree depth sweep — scanning depth 4 to 15...")
print(f"{'Depth':<8} {'F1':>8} {'Size (KB)':>12} {'Size (MB)':>10}")
print("-" * 42)

dt_sweep_results = []

for depth in range(4, 16):
    dt = DecisionTreeClassifier(
        max_depth=depth,
        criterion='gini',
        random_state=42
    )
    dt.fit(X_train, y_train_enc)
    y_pred = dt.predict(X_test)

    f1 = f1_score(y_test_enc, y_pred, average='weighted', zero_division=0)

    # Measure size
    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        tmp = f.name
    joblib.dump(dt, tmp, compress=0)
    size_kb = os.path.getsize(tmp) / 1024
    os.unlink(tmp)

    dt_sweep_results.append({
        'depth': depth, 'f1': f1, 'size_kb': size_kb
    })

    print(f"  {depth:<6} {f1:>8.4f} {size_kb:>12.1f} {size_kb/1024:>10.3f}")

# Find the elbow — best F1 per KB (efficiency score)
for r in dt_sweep_results:
    r['efficiency'] = r['f1'] / (r['size_kb'] / 1000)

best_dt_config = max(dt_sweep_results, key=lambda x: x['efficiency'])
print(f"\nBest depth by F1/size efficiency: {best_dt_config['depth']}")
print(f"  F1={best_dt_config['f1']:.4f}  "
      f"Size={best_dt_config['size_kb']:.1f} KB")

Train and benchmarking final Decision Tree

In [ ]:
print("Training final Decision Tree at depth=8...")

t0 = time.perf_counter()
best_dt = DecisionTreeClassifier(
    max_depth=8,
    criterion='gini',
    random_state=42
)
best_dt.fit(X_train, y_train_enc)
dt_train_time = time.perf_counter() - t0

print(f"Training time: {dt_train_time:.2f}s")

# Save model
dt_path = os.path.join(MODELS_PATH, "decision_tree_d8.joblib")
joblib.dump(best_dt, dt_path)
print(f"Saved to: {dt_path}")

# Full benchmark
dt_benchmark = benchmark_model(
    name="Decision Tree depth=8 (Ours)",
    model=best_dt,
    X_test=X_test,
    y_test_enc=y_test_enc,
    le=le,
    source="our_model"
)
dt_benchmark["training_time_s"] = dt_train_time
dt_benchmark["depth"] = 8
dt_benchmark["n_features"] = 10

# Print classification report for full per-class detail
print("\nFull classification report:")
y_pred_dt = best_dt.predict(X_test)
print(classification_report(
    y_test_enc, y_pred_dt,
    target_names=le.classes_, digits=4
))



Balanced DT


In [ ]:
print("Training Decision Tree with balanced class weights...")

t0 = time.perf_counter()
best_dt_balanced = DecisionTreeClassifier(
    max_depth=8,
    criterion='gini',
    class_weight='balanced',   # only change
    random_state=42
)
best_dt_balanced.fit(X_train, y_train_enc)
dt_balanced_train_time = time.perf_counter() - t0

print(f"Training time: {dt_balanced_train_time:.2f}s")

dt_balanced_path = os.path.join(MODELS_PATH, "decision_tree_d8_balanced.joblib")
joblib.dump(best_dt_balanced, dt_balanced_path)

dt_balanced_benchmark = benchmark_model(
    name="Decision Tree depth=8 balanced (Ours)",
    model=best_dt_balanced,
    X_test=X_test,
    y_test_enc=y_test_enc,
    le=le,
    source="our_model"
)
dt_balanced_benchmark["training_time_s"] = dt_balanced_train_time
dt_balanced_benchmark["depth"] = 8
dt_balanced_benchmark["note"] = "class_weight=balanced"

print("\nFull classification report:")
y_pred_bal = best_dt_balanced.predict(X_test)
print(classification_report(
    y_test_enc, y_pred_bal,
    target_names=le.classes_, digits=4
))

# 7. Addressing Class Imbalance
Using SMOTE (Synthetic Minority Over-sampling Technique) to ensure the model captures minority attack classes effectively.

In [ ]:
from imblearn.over_sampling import SMOTE

print("Class distribution before SMOTE:")
for i, cls in enumerate(le.classes_):
    count = (y_train_enc == i).sum()
    print(f"  {cls:<20} {count:>7,}")

# Bring minority classes up to 10,000 minimum
# Only upsample — never touch majority classes
sampling_strategy = {}
target_minimum = 10000
for i in range(len(le.classes_)):
    current = int((y_train_enc == i).sum())
    if current < target_minimum:
        sampling_strategy[i] = target_minimum

print(f"\nSMOTE will upsample: { {le.classes_[k]: v for k, v in sampling_strategy.items()} }")

smote = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=5       # removed n_jobs
)

print("\nRunning SMOTE...")
t0 = time.perf_counter()
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train_enc)
smote_time = time.perf_counter() - t0

print(f"Done in {smote_time:.1f}s")
print(f"\nClass distribution after SMOTE:")
for i, cls in enumerate(le.classes_):
    before = int((y_train_enc == i).sum())
    after  = int((y_train_sm == i).sum())
    added  = after - before
    if added > 0:
        print(f"  {cls:<20} {before:>7,} → {after:>7,}  (+{added:,})")
    else:
        print(f"  {cls:<20} {before:>7,} → {after:>7,}")

print(f"\nTraining set size: {len(X_train):,} → {len(X_train_sm):,}")
print(f"Test set untouched: {len(X_test):,} rows")
print("\n✓ Test set was NOT touched — benchmark integrity maintained")

Running DT again and comparing

In [ ]:
print("Training Decision Tree depth=8 on SMOTE-balanced data...")

t0 = time.perf_counter()
dt_smote = DecisionTreeClassifier(
    max_depth=8,
    criterion='gini',
    random_state=42
    # no class_weight — letting SMOTE do the balancing instead
)
dt_smote.fit(X_train_sm, y_train_sm)
dt_smote_train_time = time.perf_counter() - t0

print(f"Training time: {dt_smote_train_time:.2f}s")

dt_smote_path = os.path.join(MODELS_PATH, "decision_tree_d8_smote.joblib")
joblib.dump(dt_smote, dt_smote_path)
print(f"Saved to: {dt_smote_path}")

dt_smote_benchmark = benchmark_model(
    name="Decision Tree depth=8 SMOTE (Ours)",
    model=dt_smote,
    X_test=X_test,
    y_test_enc=y_test_enc,
    le=le,
    source="our_model"
)
dt_smote_benchmark["training_time_s"] = dt_smote_train_time
dt_smote_benchmark["depth"] = 8
dt_smote_benchmark["note"] = "SMOTE upsampled Bot/BruceForce/WebAttack to 10k"

print("\nFull classification report:")
y_pred_smote = dt_smote.predict(X_test)
print(classification_report(
    y_test_enc, y_pred_smote,
    target_names=le.classes_, digits=4
))

# Quick comparison vs original DT
print("\n── Minority class comparison ──────────────────────────")
print(f"{'Class':<20} {'Original F1':>12} {'Post-SMOTE F1':>14}")
print("-" * 48)
orig_report   = f1_score(y_test_enc,
                         best_dt.predict(X_test),
                         average=None, zero_division=0)
smote_report  = f1_score(y_test_enc,
                         y_pred_smote,
                         average=None, zero_division=0)
for i, cls in enumerate(le.classes_):
    orig  = orig_report[i]
    after = smote_report[i]
    delta = after - orig
    arrow = "↑" if delta > 0.001 else ("↓" if delta < -0.001 else "─")
    print(f"  {cls:<18} {orig:>12.4f} {after:>14.4f}  "
          f"{arrow} {abs(delta):+.4f}")

# 8. Advanced Ensemble: XGBoost with Optuna
Optimizing a high-performance XGBoost model using Bayesian search with a penalty for model size.

In [ ]:
import optuna
from optuna.samplers import TPESampler

# Use SMOTE data for all remaining models
# XGBoost needs float32 for GPU
X_train_xgb = X_train_sm.astype(np.float32)
X_test_xgb  = X_test.astype(np.float32)

print(f"Training data: {X_train_xgb.shape}")
print(f"Test data:     {X_test_xgb.shape}")
print(f"Device: GPU (cuda)")

def xgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 300),
        "max_depth":         trial.suggest_int("max_depth", 3, 6),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "objective":         "multi:softmax",
        "num_class":         len(le.classes_),
        "tree_method":       "hist",
        "device":            "cuda",
        "random_state":      42,
        "verbosity":         0,
    }

    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train_xgb, y_train_sm,
        eval_set=[(X_test_xgb, y_test_enc)],
        verbose=False
    )
    y_pred = model.predict(X_test_xgb)
    f1 = f1_score(y_test_enc, y_pred, average='weighted', zero_division=0)

    # Size penalty — discourage models over 1MB
    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        tmp = f.name
    joblib.dump(model, tmp, compress=0)
    size_kb = os.path.getsize(tmp) / 1024
    os.unlink(tmp)

    # Penalise F1 if model exceeds 1MB
    if size_kb > 1024:
        f1 = f1 * (1024 / size_kb)

    return f1

print("Starting XGBoost Optuna sweep — 50 trials...")
print("This will take approximately 10-20 minutes on GPU.\n")

xgb_study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\n{'='*55}")
print(f"XGBoost Optuna complete")
print(f"  Best F1 (penalised): {xgb_study.best_value:.4f}")
print(f"  Best params:")
for k, v in xgb_study.best_params.items():
    print(f"    {k:<22} {v}")

Train final XGBoost on best params

In [ ]:
print("Training final XGBoost on best Optuna params...")

best_xgb_params = xgb_study.best_params.copy()
best_xgb_params.update({
    "objective":   "multi:softmax",
    "num_class":   len(le.classes_),
    "tree_method": "hist",
    "device":      "cuda",
    "random_state": 42,
    "verbosity":   0,
})

t0 = time.perf_counter()
best_xgb = xgb.XGBClassifier(**best_xgb_params)
best_xgb.fit(X_train_xgb, y_train_sm, verbose=False)
xgb_train_time = time.perf_counter() - t0

print(f"Training time: {xgb_train_time:.2f}s")

xgb_path = os.path.join(MODELS_PATH, "xgboost_optuna.joblib")
joblib.dump(best_xgb, xgb_path)
print(f"Saved to: {xgb_path}")

xgb_benchmark = benchmark_model(
    name="XGBoost Optuna (Ours)",
    model=best_xgb,
    X_test=X_test_xgb,
    y_test_enc=y_test_enc,
    le=le,
    source="our_model"
)
xgb_benchmark["training_time_s"]  = xgb_train_time
xgb_benchmark["optuna_trials"]    = 50
xgb_benchmark["best_params"]      = xgb_study.best_params
xgb_benchmark["note"] = "SMOTE + Optuna TPE 50 trials + GPU"

print("\nFull classification report:")
y_pred_xgb = best_xgb.predict(X_test_xgb)
print(classification_report(
    y_test_enc, y_pred_xgb,
    target_names=le.classes_, digits=4
))

# Comparison vs DT
print("\n── vs Decision Tree (original) ────────────────────────")
print(f"{'Class':<20} {'DT F1':>8} {'XGB F1':>8} {'Delta':>8}")
print("-" * 48)
dt_f1s  = f1_score(y_test_enc, best_dt.predict(X_test),
                   average=None, zero_division=0)
xgb_f1s = f1_score(y_test_enc, y_pred_xgb,
                   average=None, zero_division=0)
for i, cls in enumerate(le.classes_):
    delta = xgb_f1s[i] - dt_f1s[i]
    arrow = "↑" if delta > 0.001 else ("↓" if delta < -0.001 else "─")
    print(f"  {cls:<18} {dt_f1s[i]:>8.4f} {xgb_f1s[i]:>8.4f}  "
          f"{arrow} {delta:+.4f}")

# 9. Comparative Ensemble: LightGBM
Training a LightGBM model to compare efficiency and accuracy against XGBoost and Decision Trees.

In [ ]:
def lgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 150),
        "max_depth":         trial.suggest_int("max_depth", 4, 6),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 31),
        "learning_rate":     trial.suggest_float("learning_rate", 0.05, 0.2),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 150),
        "subsample":         trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "objective":         "multiclass",
        "num_class":         len(le.classes_),
        "device":            "cpu",
        "n_jobs":            -1,
        "random_state":      42,
        "verbosity":         -1,
        "force_col_wise":    True,
        "min_data_in_leaf":  50,
    }

    try:
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train_xgb, y_train_sm,
            eval_set=[(X_test_xgb, y_test_enc)],
            callbacks=[lgb.early_stopping(10, verbose=False),
                       lgb.log_evaluation(period=-1)]
        )
    except Exception:
        return 0.0

    y_pred = model.predict(X_test_xgb)
    f1 = f1_score(y_test_enc, y_pred, average='weighted', zero_division=0)

    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        tmp = f.name
    joblib.dump(model, tmp, compress=0)
    size_kb = os.path.getsize(tmp) / 1024
    os.unlink(tmp)

    if size_kb > 1024:
        f1 = f1 * (1024 / size_kb)

    return f1

print("Starting LightGBM Optuna sweep — 15 trials (tightened search)...")

lgb_study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)
lgb_study.optimize(lgb_objective, n_trials=15, show_progress_bar=True)

print(f"\nLightGBM Optuna complete")
print(f"  Best F1: {lgb_study.best_value:.4f}")
print(f"  Best params:")
for k, v in lgb_study.best_params.items():
    print(f"    {k:<25} {v}")

In [ ]:
print("Training final LightGBM on best Optuna params...")

X_train_lgb = X_train_xgb
X_test_lgb = X_test_xgb

best_lgb_params = lgb_study.best_params.copy()
best_lgb_params.update({
    "objective":        "multiclass",
    "num_class":        len(le.classes_),
    "device":           "cpu",
    "n_jobs":           -1,
    "random_state":     42,
    "verbosity":        -1,
    "force_col_wise":   True,
    "min_data_in_leaf": 50,
})

t0 = time.perf_counter()
best_lgb = lgb.LGBMClassifier(**best_lgb_params)
best_lgb.fit(
    X_train_lgb, y_train_sm,
    eval_set=[(X_test_lgb, y_test_enc)],
    callbacks=[lgb.early_stopping(20, verbose=False),
               lgb.log_evaluation(period=-1)]
)
lgb_train_time = time.perf_counter() - t0

print(f"Training time: {lgb_train_time:.2f}s")

lgb_path = os.path.join(MODELS_PATH, "lightgbm_optuna.joblib")
joblib.dump(best_lgb, lgb_path)
print(f"Saved to: {lgb_path}")

lgb_benchmark = benchmark_model(
    name="LightGBM Optuna (Ours)",
    model=best_lgb,
    X_test=X_test_lgb,
    y_test_enc=y_test_enc,
    le=le,
    source="our_model"
)
lgb_benchmark["training_time_s"] = lgb_train_time
lgb_benchmark["optuna_trials"]   = 50
lgb_benchmark["best_params"]     = lgb_study.best_params
lgb_benchmark["note"] = "SMOTE + Optuna TPE 50 trials + CPU (GPU avoided due to T4 split bug)"

print("\nFull classification report:")
y_pred_lgb = best_lgb.predict(X_test_lgb)
print(classification_report(
    y_test_enc, y_pred_lgb,
    target_names=le.classes_, digits=4
))

# Three-way comparison
print("\n── Three-way comparison ───────────────────────────────")
print(f"{'Class':<20} {'DT':>7} {'XGB':>7} {'LGB':>7}")
print("-" * 45)
dt_f1s  = f1_score(y_test_enc, best_dt.predict(X_test),
                   average=None, zero_division=0)
xgb_f1s = f1_score(y_test_enc, best_xgb.predict(X_test_xgb),
                   average=None, zero_division=0)
lgb_f1s = f1_score(y_test_enc, y_pred_lgb,
                   average=None, zero_division=0)

for i, cls in enumerate(le.classes_):
    print(f"  {cls:<18} {dt_f1s[i]:>7.4f} "
          f"{xgb_f1s[i]:>7.4f} {lgb_f1s[i]:>7.4f}")

print(f"\n  {'Overall F1':<18} "
      f"{f1_score(y_test_enc, best_dt.predict(X_test), average='weighted'):>7.4f} "
      f"{f1_score(y_test_enc, best_xgb.predict(X_test_xgb), average='weighted'):>7.4f} "
      f"{f1_score(y_test_enc, y_pred_lgb, average='weighted'):>7.4f}")

print(f"\n  {'Size (KB)':<18} "
      f"{dt_benchmark['model_size_kb']:>7.1f} "
      f"{xgb_benchmark['model_size_kb']:>7.1f} "
      f"{lgb_benchmark['model_size_kb']:>7.1f}")

print(f"\n  {'Latency (ms)':<18} "
      f"{dt_benchmark['latency_single_ms']:>7.3f} "
      f"{xgb_benchmark['latency_single_ms']:>7.3f} "
      f"{lgb_benchmark['latency_single_ms']:>7.3f}")

print(f"\n  {'Throughput (K/s)':<18} "
      f"{dt_benchmark['throughput_samples_per_sec']/1000:>7.0f} "
      f"{xgb_benchmark['throughput_samples_per_sec']/1000:>7.0f} "
      f"{lgb_benchmark['throughput_samples_per_sec']/1000:>7.0f}")

# 10. Final Comparison & Visualization
Aggregating results from all models and generating visual benchmarks against reference implementations.

In [ ]:
# Collect all three model benchmarks
your_benchmarks = [
    dt_benchmark,
    xgb_benchmark,
    lgb_benchmark
]

# Save to features folder so it sits alongside ref1 and ref2 JSONs
output_path = os.path.join(FEATURES_PATH, "your_benchmarks.json")
with open(output_path, "w") as f:
    json.dump(your_benchmarks, f, indent=2)

print(f"✓ Saved your_benchmarks.json")
print(f"  Path: {output_path}")
print(f"  Models: {[b['model'] for b in your_benchmarks]}")

# Also save a combined summary table
print("\n── Final summary ──────────────────────────────────────────")
print(f"{'Model':<35} {'Size(KB)':>9} {'F1':>7} {'Lat(ms)':>9} {'Thru(K/s)':>11}")
print("-" * 75)
for b in your_benchmarks:
    print(f"  {b['model']:<33} "
          f"{b['model_size_kb']:>9.1f} "
          f"{b['f1_weighted']:>7.4f} "
          f"{b['latency_single_ms']:>9.3f} "
          f"{b['throughput_samples_per_sec']/1000:>11.0f}")

Load all JSONs and generate comparison graphs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load all three benchmark files ───────────────────────────────────
ref1_path = os.path.join(FEATURES_PATH, "ref1_benchmarks.json")
ref2_path = os.path.join(FEATURES_PATH, "ref2_benchmarks.json")
your_path = os.path.join(FEATURES_PATH, "your_benchmarks.json")

with open(ref1_path) as f: ref1 = json.load(f)
with open(ref2_path) as f: ref2 = json.load(f)
with open(your_path) as f: ours = json.load(f)

# ── Flatten into one list, tag each with group ───────────────────────
all_models = []
for b in ref1:
    b['group'] = 'Reference 1'
    all_models.append(b)
for b in ref2:
    b['group'] = 'Reference 2'
    all_models.append(b)
for b in ours:
    b['group'] = 'Ours'
    all_models.append(b)

# Short display names
name_map = {
    "Random Forest (Ref1)":                    "RF (Ref1)",
    "XGBoost (Ref1)":                          "XGB (Ref1)",
    "Random Forest 15-trees depth=8 (Ref2)":  "RF (Ref2)",
    "Decision Tree depth=8 (Ref2)":            "DT (Ref2)",
    "KNN k=8 (Ref2)":                          "KNN (Ref2)",
    "Decision Tree depth=8 (Ours)":            "DT (Ours)",
    "XGBoost Optuna (Ours)":                   "XGB (Ours)",
    "LightGBM Optuna (Ours)":                  "LGB (Ours)",
}
for b in all_models:
    b['short_name'] = name_map.get(b['model'], b['model'])

# ── Color scheme ─────────────────────────────────────────────────────
group_colors = {
    'Reference 1': '#E05C5C',
    'Reference 2': '#F4A942',
    'Ours':        '#0E9AA7',
}

def get_colors(models):
    return [group_colors[m['group']] for m in models]

labels = [m['short_name'] for m in all_models]
colors = get_colors(all_models)
x = np.arange(len(all_models))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('IDS Model Benchmark — All Models Compared',
             fontsize=16, fontweight='bold', y=1.01)

# ── Chart 1: Model Size ───────────────────────────────────────────────
ax1 = axes[0, 0]
sizes = [m['model_size_kb'] for m in all_models]
bars = ax1.bar(x, sizes, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_title('Model Size (KB) — lower is better',
              fontweight='bold', fontsize=12)
ax1.set_ylabel('Size (KB)')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax1.axhline(y=1024, color='gray', linestyle='--',
            alpha=0.5, label='1 MB threshold')
ax1.legend(fontsize=9)
for bar, val in zip(bars, sizes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{val:.0f}', ha='center', va='bottom', fontsize=8)
ax1.set_yscale('log')
ax1.grid(axis='y', alpha=0.3)

# ── Chart 2: Weighted F1 ──────────────────────────────────────────────
ax2 = axes[0, 1]
f1s = [m['f1_weighted'] for m in all_models]
bars2 = ax2.bar(x, f1s, color=colors, edgecolor='white', linewidth=0.5)
ax2.set_title('Weighted F1 Score — higher is better',
              fontweight='bold', fontsize=12)
ax2.set_ylabel('Weighted F1')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax2.set_ylim(0.92, 1.001)
for bar, val in zip(bars2, f1s):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.0005,
             f'{val:.4f}', ha='center', va='bottom', fontsize=7.5)
ax2.grid(axis='y', alpha=0.3)

# ── Chart 3: Single-sample Latency ───────────────────────────────────
ax3 = axes[1, 0]
lats = [m['latency_single_ms'] for m in all_models]
bars3 = ax3.bar(x, lats, color=colors, edgecolor='white', linewidth=0.5)
ax3.set_title('Single-Sample Latency (ms) — lower is better',
              fontweight='bold', fontsize=12)
ax3.set_ylabel('Latency (ms)')
ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax3.axhline(y=10, color='red', linestyle='--',
            alpha=0.6, label='10ms IoV requirement')
ax3.legend(fontsize=9)
for bar, val in zip(bars3, lats):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.3,
             f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax3.grid(axis='y', alpha=0.3)

# ── Chart 4: Size vs F1 Scatter ───────────────────────────────────────
ax4 = axes[1, 1]
for b in all_models:
    ax4.scatter(
        b['model_size_kb'], b['f1_weighted'],
        color=group_colors[b['group']],
        s=120, zorder=5,
        edgecolors='white', linewidth=0.8
    )
    ax4.annotate(
        b['short_name'],
        (b['model_size_kb'], b['f1_weighted']),
        textcoords="offset points",
        xytext=(6, 3), fontsize=8
    )

ax4.set_title('Model Size vs F1',
              fontweight='bold', fontsize=12)
ax4.set_xlabel('Model Size (KB) — log scale')
ax4.set_ylabel('Weighted F1')
ax4.set_xscale('log')
ax4.set_ylim(0.92, 1.001)
ax4.grid(alpha=0.3)
ax4.axvline(x=1024, color='gray', linestyle='--',
            alpha=0.5, label='1 MB')
ax4.legend(fontsize=9)

# ── Legend for all charts ─────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=group_colors['Reference 1'], label='Reference 1'),
    mpatches.Patch(color=group_colors['Reference 2'], label='Reference 2'),
    mpatches.Patch(color=group_colors['Ours'],        label='Our models'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=3, fontsize=11, frameon=True,
           bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()

# Save to Drive
graph_path = os.path.join(BASE_PATH, "benchmark_comparison.png")
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Graph saved to: {graph_path}")

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np

# Binarize test labels for OvR ROC AUC
n_classes = len(le.classes_)
y_test_bin = label_binarize(y_test_enc, classes=list(range(n_classes)))

print("Computing ROC AUC scores (macro OvR)...")
print("This requires predict_proba — may take a moment on full test set.\n")

auc_results = {}

# Decision Tree
y_prob_dt = best_dt.predict_proba(X_test)
auc_dt = roc_auc_score(y_test_bin, y_prob_dt, multi_class="ovr", average="macro")
auc_results["Decision Tree depth=8 (Ours)"] = auc_dt
print(f"Decision Tree ROC AUC (macro OvR): {auc_dt:.4f}")

# XGBoost
y_prob_xgb = best_xgb.predict_proba(X_test_xgb)
auc_xgb = roc_auc_score(y_test_bin, y_prob_xgb, multi_class="ovr", average="macro")
auc_results["XGBoost Optuna (Ours)"] = auc_xgb
print(f"XGBoost ROC AUC (macro OvR):       {auc_xgb:.4f}")

# LightGBM
y_prob_lgb = best_lgb.predict_proba(X_test_lgb)
auc_lgb = roc_auc_score(y_test_bin, y_prob_lgb, multi_class="ovr", average="macro")
auc_results["LightGBM Optuna (Ours)"] = auc_lgb
print(f"LightGBM ROC AUC (macro OvR):      {auc_lgb:.4f}")

print(f"\nAll ROC AUC scores: {auc_results}")

# Update and re-save your_benchmarks.json with AUC added
import json, os

benchmark_path = os.path.join(FEATURES_PATH, "your_benchmarks.json")
with open(benchmark_path) as f:
    benchmarks = json.load(f)

for b in benchmarks:
    if b["model"] in auc_results:
        b["roc_auc_macro_ovr"] = round(auc_results[b["model"]], 4)

with open(benchmark_path, "w") as f:
    json.dump(benchmarks, f, indent=2)

print(f"\n✓ your_benchmarks.json updated with ROC AUC scores")



# 11. ROC AUC Analysis
Evaluating class separation performance using Receiver Operating Characteristic (ROC) curves.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Binarize labels
n_classes = len(le.classes_)
y_test_bin = label_binarize(y_test_enc, classes=list(range(n_classes)))

def plot_ovr_roc(ax, y_true_bin, y_score, model_name):
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    # Per-class ROC
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Macro-average ROC
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)

    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

    mean_tpr /= n_classes

    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # Plot macro curve only (cleaner for poster)
    ax.plot(
        fpr["macro"],
        tpr["macro"],
        label=f"{model_name} (AUC = {roc_auc['macro']:.4f})",
        linewidth=2
    )

# Get probabilities
y_prob_dt = best_dt.predict_proba(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test_xgb)
y_prob_lgb = best_lgb.predict_proba(X_test_lgb)

# Plot
fig, ax = plt.subplots(figsize=(7, 6))

plot_ovr_roc(ax, y_test_bin, y_prob_dt, "Decision Tree")
plot_ovr_roc(ax, y_test_bin, y_prob_xgb, "XGBoost")
plot_ovr_roc(ax, y_test_bin, y_prob_lgb, "LightGBM")

# Diagonal baseline
ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (Macro-Averaged, OvR)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("roc_curves_macro.png", dpi=200)
plt.show()